# The research record, in figures

Every figure below is drawn from the tables in `tables/` alone; no survey data are needed.
Intervals are 95%, from resampling whole survey clusters of the held-out women.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
T = "tables/"
cells = pd.read_csv(T + "library_cells.csv")
tuned = cells[(cells.status == "done") & (cells.variant == "tuned")]
RUNGS = ["anc1plus", "anc4plus", "anc8plus"]
SETS = {"pool": "Full pool", "s21": "The 21", "short": "Short set", "pars": "Parsimonious"}

## Every tuned model on every set, held-out average precision

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r in zip(axes, RUNGS):
    t = tuned[tuned.rung == r].pivot_table(index="family", columns="set", values="held_AP")[list(SETS)]
    im = ax.imshow(t.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(SETS)), SETS.values(), rotation=30); ax.set_yticks(range(len(t)), t.index)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.values[i, j]
            if np.isfinite(v): ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
    ax.set_title(r.upper().replace("PLUS", "+"))
plt.tight_layout(); plt.show()

## More predictors against the 21 (paired difference, each model)

In [ ]:
ps = pd.read_csv(T + "predictor_sets.csv").dropna(subset=["AP_minus_21"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, r in zip(axes, RUNGS):
    g = ps[ps.rung == r]
    for k, (s, lab) in enumerate([(s, l) for s, l in SETS.items() if s != "s21"]):
        h = g[g.set == s]
        x = k + np.linspace(-0.3, 0.3, len(h))
        ax.errorbar(x, h.AP_minus_21, yerr=[h.AP_minus_21 - h.AP_minus_21_lo, h.AP_minus_21_hi - h.AP_minus_21], fmt="o", ms=3, lw=0.8)
    ax.axhline(0, color="grey", lw=0.8); ax.set_xticks(range(3), [l for s, l in SETS.items() if s != "s21"])
    ax.set_title(r.upper().replace("PLUS", "+"))
axes[0].set_ylabel("AP minus the same model on the 21")
plt.tight_layout(); plt.show()

## Recalibration: what isotonic and Platt did

In [ ]:
cal = pd.read_csv(T + "calibration.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for m, c in (("isotonic", "tab:red"), ("platt", "tab:blue")):
    h = cal[cal.method == m]
    axes[0].hist(h.AP_difference, bins=30, alpha=0.6, label=m, color=c)
    axes[1].scatter(h.slope_tuned, h.slope_recalibrated, s=10, alpha=0.6, label=m, color=c)
axes[0].axvline(0, color="grey"); axes[0].set_xlabel("held-out AP, recalibrated minus tuned"); axes[0].legend()
axes[1].plot([0, 2], [0, 2], color="grey", lw=0.8); axes[1].axhline(1, color="grey", ls=":"); axes[1].axvline(1, color="grey", ls=":")
axes[1].set_xlabel("calibration slope, tuned"); axes[1].set_ylabel("calibration slope, recalibrated"); axes[1].legend()
plt.tight_layout(); plt.show()

## Ensembles and networks against the best single model

In [ ]:
ens = pd.read_csv(T + "ensembles.csv"); net = pd.read_csv(T + "networks.csv")
rows = [(f"{e.ensemble} ({e.lineup})", e) for e in ens.itertuples()] + [(n.network, n) for n in net.itertuples()]
lab = sorted({k for k, _ in rows})
fig, ax = plt.subplots(figsize=(10, 4))
for i, k in enumerate(lab):
    vals = [r for kk, r in rows if kk == k]
    x = i + np.linspace(-0.3, 0.3, len(vals))
    ax.errorbar(x, [v.AP_difference for v in vals], yerr=[[v.AP_difference - v.AP_difference_lo for v in vals], [v.AP_difference_hi - v.AP_difference for v in vals]], fmt="o", ms=3, lw=0.8)
ax.axhline(0, color="grey"); ax.set_xticks(range(len(lab)), lab, rotation=20, ha="right")
ax.set_ylabel("held-out AP minus the best single model"); plt.tight_layout(); plt.show()

## Tuning: notebook 2's grids against the longer search

In [ ]:
g = pd.read_csv(T + "tuning/grid_vs_search.csv")
print(g[["rung", "model", "grid_settings", "search_trials", "grid_held_AP", "search_held_AP", "held_AP_difference"]].round(4).to_string(index=False))

## CatBoost: what finished

In [ ]:
cb = pd.read_csv(T + "catboost.csv")
print(cb.groupby(["set", "variant"]).status.value_counts().unstack(fill_value=0).to_string())
print()
print(cb[cb.status == "done"][["set", "rung", "variant", "held_AP", "held_AP_lo", "held_AP_hi"]].round(3).to_string(index=False))